# 01 — Genre Classification
**Spotify Data Mining | CISC 4631 | Group 3**

**Research Question:** Can audio features predict song genre?

**Approach:** Logistic Regression (linear baseline) vs. Random Forest (non-linear). Compare accuracy,
F1 scores, and feature importance.

> **Prerequisite:** Run `00_data_setup.ipynb` first to generate `data/df_genre_balanced.csv`.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

# Shared constants (keep in sync with 00_data_setup.ipynb)
SEED            = 42
DRIVE_DATA_PATH = '/content/drive/MyDrive/data-mining-spotify-team3/cleanedData'
ALL_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'key', 'mode'
]
np.random.seed(SEED)

## 1. Load Data

In [ ]:
import os
df = pd.read_csv(os.path.join(DRIVE_DATA_PATH, 'df_genre_balanced.csv'))
print(f'Loaded: {df.shape}')
print('\nGenre counts:')
print(df['genre'].value_counts())
df.head()

## 2. Train / Test Split

In [ ]:
X = df[ALL_FEATURES]
y = df['genre']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print('Test class distribution:')
print(y_test.value_counts())

## 3. Models

### 3.1 Logistic Regression (Baseline)

In [ ]:
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=SEED))
])

lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_lr))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr))

# Cross-validation
cv_lr = cross_val_score(lr_model, X, y, cv=10, scoring='accuracy')
print("CV Accuracy: %0.2f (+/- %0.2f)" % (cv_lr.mean(), cv_lr.std() * 2))

### 3.2 Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=SEED,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print('Random Forest Accuracy:', accuracy_score(y_test, y_pred_rf))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf))

# Cross-validation
cv_rf = cross_val_score(rf_model, X, y, cv=10, scoring='accuracy')
print("CV Accuracy: %0.2f (+/- %0.2f)" % (cv_rf.mean(), cv_rf.std() * 2))

## 4. Results

### 4.1 Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Macro F1': [
        f1_score(y_test, y_pred_lr, average='macro'),
        f1_score(y_test, y_pred_rf, average='macro')
    ],
    'Weighted F1': [
        f1_score(y_test, y_pred_lr, average='weighted'),
        f1_score(y_test, y_pred_rf, average='weighted')
    ]
})

print(results.to_string(index=False))

In [ ]:
results_pct = results.copy()
results_pct[['Accuracy', 'Macro F1', 'Weighted F1']] *= 100

ax = results_pct.plot(x='Model', y=['Accuracy', 'Macro F1', 'Weighted F1'], kind='bar', figsize=(8, 5))
plt.title('Model Performance Comparison')
plt.ylabel('Score (%)')
plt.xticks(rotation=0)
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

### 4.2 Random Forest Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': ALL_FEATURES,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

feature_importance.plot(kind='bar', x='Feature', y='Importance', legend=False, figsize=(10, 5))
plt.title('Random Forest Feature Importance')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(feature_importance.to_string(index=False))

### 4.3 Confusion Matrix (Random Forest)

In [ ]:
disp = ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf,
    display_labels=rf_model.classes_,
    cmap='Blues',
    xticks_rotation=45
)
disp.ax_.set_title('Random Forest Confusion Matrix')
plt.tight_layout()
plt.show()

## 5. Summary

Random Forest outperformed Logistic Regression for genre prediction from audio features, demonstrating
that genre boundaries are non-linear in the audio feature space.

**Key findings:**
- Hip-Hop and Electronic were the most distinguishable genres (high F1) — their audio profiles are distinct.
- Folk and Pop had the lowest F1 — their audio patterns overlap heavily with other genres.
- Country had decent recall — the model found Country songs even if sometimes it misclassified others as Country.
- Top features: `speechiness`, `danceability`, `acousticness`, `energy`, `valence`.
- Least useful: `key`, `mode` — musical key is not genre-specific.

**Baseline accuracy for 9-class balanced problem is 11.1% (random chance). Both models significantly exceed this.**